# StoryWeaver — LangChain 기반 스토리텔링 AI

캐릭터의 성격·과거·트라우마·스킬, 세계관, 인물 관계, 필연을 도구로 조회하고 여러 장면의 기억을 이어서 대본을 생성하는 과제용 노트북입니다.

### 왜 만들었는가
영화·웹툰·게임은 연재가 길어질수록 캐릭터의 성격과 말투, 능력의 한계, 인물 관계, 과거 사건 같은 설정이 계속 쌓입니다. 작가는 새로운 장면을 쓸 때마다 방대한 설정집과 이전 회차를 다시 확인해야 하며, 제작 일정이 촉박하면 캐릭터에 맞지 않는 대사나 앞뒤가 맞지 않는 행동, 관계와 사건의 연결이 끊기는 오류가 생기기 쉽습니다. StoryWeaver는 누적된 설정을 놓치지 않으면서 장면 초안을 빠르게 작성할 수 있도록 돕기 위해 만들었습니다.

### 페인포인트
- 연재 분량이 길어질수록 설정과 사건 기록이 빠르게 늘어나 필요한 내용을 찾는 데 시간이 오래 걸립니다.
- 캐릭터의 성격과 말투에 맞지 않는 대사나, 정해진 능력의 한계를 벗어난 행동이 들어갈 수 있습니다.
- 이전 장면의 감정, 부상, 비밀과 관계 변화를 놓치면 이야기의 흐름이 자연스럽게 이어지지 않습니다.
- 여러 작가가 함께 작업하면 같은 캐릭터와 세계관을 서로 다르게 해석해 작품의 일관성이 흔들릴 수 있습니다.
- 초안을 작성한 뒤 설정 오류를 찾아 수정하는 반복 작업 때문에 실제 연출과 플롯 구상에 쓸 시간이 줄어듭니다.

### 해결 방법과 비즈니스 가치
StoryWeaver에 캐릭터·세계관·관계·이전 장면을 한 번 등록하면, 작가는 이번 장면에서 일어날 상황만 입력해 설정에 맞는 대본 초안을 빠르게 얻을 수 있습니다. 초안은 누적된 설정과 비교해 한 번 더 검수되고, 장면에서 달라진 관계와 사건은 다음 장면에 이어집니다. 이를 통해 설정 검색과 수정에 드는 시간을 줄이고 대본 제작 속도를 높여, 영화사·웹툰 스튜디오·게임 제작팀이 더 많은 장면을 안정적으로 생산할 수 있습니다. 작가는 반복적인 설정 확인보다 핵심 플롯과 연출, 감정 표현에 집중할 수 있습니다.

### 사용한 LangChain 컴포넌트
`ChatOpenAI` · `create_agent` · `@tool` 6개 · Pydantic 구조화 출력 · `ModelRetryMiddleware` · 이전 장면 상태 조회 · 2-Agent 순차 오케스트레이션

핵심 흐름: **설정 입력 → LangChain Tool 호출 → 작가 Agent 초안 → 검수 Agent 교정 → 관계·사건·필연 갱신 → 최종 대본 출력**

## 1. 라이브러리 설치
처음 실행할 때만 설치합니다. 설치 후 커널 재시작이 필요할 수 있습니다.

In [1]:
%pip install -q -U langchain langchain-openai pydantic requests

Note: you may need to restart the kernel to use updated packages.


## StoryWeaver LangChain 아키텍처
아래는 별도 라이브러리 없이 노트북에서 바로 볼 수 있는 전체 처리 구조입니다.

```text
[사용자 입력]
- WORLD_SETTING
- CHARACTER_1 / CHARACTER_2
- SCENE_REQUEST
        │
        ▼
[WRITER_AGENT + ChatOpenAI] ◀──────────────┐
  └ ModelRetryMiddleware                   │ Tool 호출 및 결과
        │                                   │
        │ SceneOutput 초안                  │
        ▼                                   │
[REVIEWER_AGENT + ChatOpenAI] ◀────────────┤
  └ ModelRetryMiddleware                   │ 같은 설정을 독립적으로 재조회
        │                                   │
        │ ReviewOutput                      │
        ▼                                   │
[승인 또는 교정된 finalScene]              │
        │                                   │
        ├───────────────┐                   │
        ▼               ▼                   │
[대본 화면 출력]  [_apply_story_updates()] │
                    │                       │
                    ├ 사건 저장              │
                    ├ 관계 갱신              │
                    └ 필연 진행도 갱신       │
                    │                       │
                    ▼                       │
             [PROJECT + SCENES] ────────────┘
                    │
                    └ get_previous_scenes → 다음 장면 기억

[공유 Tool Layer]
1. get_character_profiles      : 캐릭터 설정 조회
2. get_relationship_state      : 관계와 사건 조회
3. search_world_lore           : 세계관 규칙 조회
4. get_inevitabilities         : 필연과 조건 조회
5. get_previous_scenes         : 이전 장면 기억 조회
6. search_historical_context   : 실제 역사 외부 검색
```

### 아키텍처 구성요소

| 계층 | 컴포넌트 | 역할 |
|---|---|---|
| 입력 | `WORLD_SETTING`, `CHARACTER_1`, `CHARACTER_2`, `SCENE_REQUEST` | 세계관·캐릭터·장면 요구사항을 입력합니다. |
| 모델 | `ChatOpenAI` 2개 | 작가 모델은 창의적인 초안을, 검수 모델은 낮은 temperature로 일관성 검사를 수행합니다. |
| Agent | `WRITER_AGENT`, `REVIEWER_AGENT` | 작가 Agent가 초안을 만들고 검수 Agent가 승인 또는 교정합니다. |
| Middleware | `ModelRetryMiddleware` | 사용자가 장면 생성 요청을 다시 입력하거나 버튼을 다시 누르지 않아도 되도록, 일시적인 연결 오류가 발생하면 내부에서 한 번 재시도합니다. |
| Tool | 캐릭터·관계·세계관·필연·이전 장면·역사 검색 6종 | Agent가 필요한 설정만 정확하게 조회합니다. 작가와 검수 Agent가 같은 Tool을 독립적으로 호출합니다. |
| Structured Output | `SceneOutput`, `ReviewOutput` | 자유 텍스트를 검증 가능한 장면 초안과 검수 결과로 변환합니다. |
| 상태 갱신 | `_apply_story_updates()` | 최종 장면의 사건, 관계 변화, 필연 진행도를 `PROJECT`와 `SCENES`에 저장합니다. Middleware가 아니라 애플리케이션 로직입니다. |
| 출력 | `show_generated_scene()`, `final_script()` | 구조화된 데이터를 읽기 쉬운 장면과 최종 대본으로 렌더링합니다. |

`PROJECT + SCENES → get_previous_scenes → Agent`로 돌아가는 화살표가 장면 기억 루프입니다. 장면 2는 장면 1을, 장면 3은 장면 1과 2를 조회하게 됩니다.

## 2. API 키 입력
키는 화면에 표시되지 않으며 노트북 파일에도 저장되지 않습니다.

In [2]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

print("API 키가 준비되었습니다.")

API 키가 준비되었습니다.


## 3. 프로젝트 데이터와 편집 함수
처음에는 빈 프로젝트입니다. 아래 함수로 세계관, 캐릭터, 필연을 추가합니다. 노트북 실행 중에는 `PROJECT`와 `SCENES`가 메모리 역할을 합니다.

In [3]:
from __future__ import annotations

import json
from uuid import uuid4

PROJECT = {
    "world": {
        "title": "제목 없는 이야기",
        "mode": "fictional",
        "era": "",
        "summary": "",
        "history": "",
        "rules": "",
        "factions": "",
    },
    "characters": [],
    "fates": [],
    "relationships": [],
    "events": [],
}
SCENES = []


def set_world(title, era, summary, history="", rules="", factions="", mode="fictional"):
    PROJECT["world"] = {
        "title": title, "mode": mode, "era": era, "summary": summary,
        "history": history, "rules": rules, "factions": factions,
    }


def add_character(name, personality, speech_style, *, age="", gender="", role="",
                  backstory="", trauma="", strengths="", weaknesses="",
                  goal="", secret="", relationships="", skills=""):
    character = {
        "id": str(uuid4()), "name": name, "age": age, "gender": gender,
        "role": role, "personality": personality, "backstory": backstory,
        "trauma": trauma, "strengths": strengths, "weaknesses": weaknesses,
        "speechStyle": speech_style, "goal": goal, "secret": secret,
        "relationships": relationships, "skills": skills,
    }
    PROJECT["characters"].append(character)
    return character["id"]


def update_character(character_id, **changes):
    character = next(c for c in PROJECT["characters"] if c["id"] == character_id)
    character.update(changes)
    return character


def add_fate(character_id, statement, *, fate_type="기타", condition="",
             deadline="", rigidity="절대적"):
    fate = {
        "id": str(uuid4()), "characterId": character_id, "type": fate_type,
        "statement": statement, "condition": condition, "deadline": deadline,
        "rigidity": rigidity, "progress": 0,
    }
    PROJECT["fates"].append(fate)
    return fate["id"]


def update_fate(fate_id, **changes):
    fate = next(f for f in PROJECT["fates"] if f["id"] == fate_id)
    fate.update(changes)
    return fate


print("빈 StoryWeaver 프로젝트를 만들었습니다.")

빈 StoryWeaver 프로젝트를 만들었습니다.


## 4. 시연용 작품 설정 입력
아래의 `WORLD_SETTING`, `CHARACTER_1`, `CHARACTER_2`가 발표자가 직접 수정하는 입력 영역입니다. 현재는 특정 영화 한 편이 아닌 스파이더맨 코믹스의 공통적인 요소를 바탕으로 한 교육용 시연 설정이 들어 있습니다. 스킬에는 초능력뿐 아니라 과학, 컴퓨터, 협상 같은 현실 능력과 사용 제약을 함께 적습니다.

In [4]:
# ===== 발표자가 수정하는 입력 영역 =====
WORLD_SETTING = {
    "title": "거미줄 너머의 선택",
    "mode": "fictional",
    "era": "현대의 뉴욕. 초인과 첨단 기업 기술이 공존하지만 영웅의 정체는 대부분 공개되지 않았다.",
    "summary": "스파이더맨이 시민의 일상과 거대 기업의 위협 사이에서 책임을 선택하는 슈퍼히어로 세계관",
    "history": "오스코프는 생명공학과 군사용 장비로 성장했다. 스파이더맨은 수년간 뉴욕을 지켰지만 언론과 시민의 평가는 엇갈린다.",
    "rules": "능력은 캐릭터 프로필에 적힌 범위에서만 사용한다. 비밀 정체를 아는 인물만 관련 정보를 말할 수 있다. 초인도 부상과 피로의 영향을 받는다.",
    "factions": "뉴욕 시민과 경찰, 데일리 뷰글, 오스코프, 거리의 자경단과 범죄 조직",
}

CHARACTER_1 = {
    "id": "peter-parker",
    "name": "피터 파커",
    "age": "20대",
    "gender": "남성",
    "role": "스파이더맨이자 젊은 과학자",
    "personality": "책임감이 강하고 다정하다. 위험할수록 농담하지만 모든 짐을 혼자 지려 한다.",
    "backstory": "거미에게 물린 뒤 초인적인 능력을 얻었고, 벤 삼촌을 잃은 일을 계기로 힘을 타인을 지키는 데 사용한다.",
    "trauma": "자신이 더 일찍 행동했다면 벤 삼촌을 구할 수 있었다는 죄책감이 있다.",
    "strengths": "공감 능력, 과학적 사고, 빠른 판단, 위험 속에서도 타인을 먼저 구하는 책임감",
    "weaknesses": "도움을 요청하지 못하고 죄책감 때문에 무모한 선택을 한다.",
    "speechStyle": "긴장을 풀기 위한 재치 있는 농담과 과학 비유를 쓰며, 누군가 위험해지면 짧고 단호하게 말한다.",
    "goal": "뉴욕과 가까운 사람들을 지키면서 피터 파커로서의 삶도 포기하지 않는다.",
    "secret": "피터 파커가 스파이더맨이라는 사실을 숨기고 있다.",
    "relationships": "노먼을 위험한 인물로 경계하지만 과학자로서의 지성과 해리의 아버지라는 위치 때문에 쉽게 공격하지 못한다.",
    "skills": "벽 타기, 초인적 근력과 반사 신경, 위험 감지, 웹 슈터, 물리학·화학·공학. 무적이 아니며 과도한 전투 후 피로해진다.",
}

CHARACTER_2 = {
    "id": "norman-osborn",
    "name": "노먼 오스본",
    "age": "50대",
    "gender": "남성",
    "role": "오스코프 수장이자 그린 고블린",
    "personality": "카리스마 있고 계산적이다. 사람을 통제 가능한 자원으로 보며 패배를 모욕으로 받아들인다.",
    "backstory": "과학과 사업으로 오스코프를 키웠고, 위험한 강화 실험과 야망을 거치며 그린 고블린이 되었다.",
    "trauma": "약함을 수치로 배운 성장 경험 때문에 두려움을 인정하지 않고 통제와 공격성으로 덮는다.",
    "strengths": "기업 권력, 과학 지식, 장기 전략, 심리 조종, 강화된 신체 능력과 고블린 장비",
    "weaknesses": "오만과 집착 때문에 상대의 선의와 예측 불가능한 희생을 계산하지 못한다.",
    "speechStyle": "침착하고 권위적으로 약점을 분석하며, 분노하면 조롱과 위협이 연극적으로 과장된다.",
    "goal": "뉴욕의 미래를 통제하고 스파이더맨의 신념을 꺾어 자신의 우월함을 증명한다.",
    "secret": "피터의 정체를 거의 확신하지만 결정적인 순간에 이용하기 위해 공개하지 않는다.",
    "relationships": "피터의 재능을 탐내며 후계자 또는 실험 대상으로 만들려 한다. 스파이더맨은 반드시 굴복시켜야 할 장애물이다.",
    "skills": "경영, 유전·화학 연구, 심리전, 강화된 근력과 회복력, 고블린 글라이더와 폭발 장비. 장비가 없으면 비행할 수 없다.",
}

INEVITABILITY = {
    "id": "fate-public-choice",
    "characterId": CHARACTER_1["id"],
    "type": "선택",
    "statement": "종막에서 피터는 정체를 지키는 것보다 시민을 구하는 선택을 해야 한다.",
    "condition": "정체 공개 위험과 대규모 인명 피해가 동시에 발생한다.",
    "deadline": "종막",
    "rigidity": "절대적",
    "progress": 0,
}

STARTING_RELATIONSHIP = {
    "id": "relationship-peter-norman",
    "characterAId": CHARACTER_1["id"],
    "characterBId": CHARACTER_2["id"],
    "past": "노먼은 피터의 과학적 재능을 높이 평가했고 피터는 노먼의 연구가 가진 위험성을 목격했다.",
    "current": "서로의 비밀을 떠보면서도 확실한 증거가 없어 겉으로는 협력한다.",
    "history": [],
}
# ===== 입력 영역 끝 =====

PROJECT["world"] = WORLD_SETTING.copy()
PROJECT["characters"] = [CHARACTER_1.copy(), CHARACTER_2.copy()]
PROJECT["fates"] = [INEVITABILITY.copy()]
PROJECT["relationships"] = [STARTING_RELATIONSHIP.copy()]
PROJECT["events"] = []
SCENES.clear()

print(f"세계관: {PROJECT['world']['title']}")
print(f"캐릭터: {', '.join(c['name'] for c in PROJECT['characters'])}")
print(f"필연: {PROJECT['fates'][0]['statement']}")

세계관: 거미줄 너머의 선택
캐릭터: 피터 파커, 노먼 오스본
필연: 종막에서 피터는 정체를 지키는 것보다 시민을 구하는 선택을 해야 한다.


## 5. 구조화된 출력(Structured Output) 스키마
LLM이 자유 형식의 긴 문장 하나를 반환하면 프로그램이 대사, 행동, 관계 변화 등을 안정적으로 구분하기 어렵습니다. StoryWeaver는 Pydantic 모델을 `create_agent(response_format=...)`에 전달하여 Agent가 정해진 필드와 자료형을 가진 결과를 반환하게 합니다. LangChain은 최종 결과를 검증한 뒤 `result["structured_response"]`에 저장하며, 필수 필드가 없거나 자료형이 맞지 않는 응답은 그대로 사용하지 않습니다.

### 각 출력 모델의 역할

| 출력 모델 | 포함하는 값 | StoryWeaver에서의 역할 |
|---|---|---|
| `DialogueLine` | 화자, 대사, 감정, 행동, 숨은 의도 | 대본 한 줄을 구성합니다. 화면에서는 대사와 연기 지시를 분리하고, 다음 장면에서는 인물이 실제로 무엇을 알고 느꼈는지 기억하는 자료가 됩니다. |
| `Continuity` | 일관성 상태, 경고, 참고 사항 | 캐릭터의 말투·능력, 이전 사건, 세계관 규칙에 충돌이 있는지 표시합니다. `consistent` 또는 `warning`으로 상태를 제한합니다. |
| `FateSignal` | 필연 ID, 진행 변화량, 이유 | 이번 장면이 등록된 필연에 얼마나 가까워졌는지 기록합니다. `progressDelta`는 0~10 범위로 제한되고 누적 진행도에 반영됩니다. |
| `RelationshipChange` | 두 캐릭터, 변화 내용, 이유 | 장면 때문에 실제로 달라진 신뢰·갈등·동맹 관계를 기록합니다. 단순히 함께 등장했다는 사실은 관계 변화로 저장하지 않습니다. |
| `StoryUpdates` | 사건 요약, 관계 변화 목록 | 생성된 장면을 다음 장면의 기억으로 바꾸는 갱신 데이터입니다. 사건 연대기와 관계 기록에 저장됩니다. |
| `SceneOutput` | 제목, 장소, 장면 설명, 대사 목록, 일관성, 필연 신호, 상태 갱신 | 작가 Agent가 반환하는 장면 전체의 최종 형식입니다. 대사는 최소 2개, 최대 24개로 제한합니다. |
| `ReviewOutput` | 승인·교정 판정, 문제 목록, 검수 요약, 최종 장면 | 검수 Agent의 출력입니다. 문제가 없으면 `approved`, 수정했으면 `revised`가 되며, 실제로 사용할 장면은 항상 `finalScene`에 담깁니다. |

### 데이터가 사용되는 흐름

```text
작가 Agent → SceneOutput 초안
             ↓
검수 Agent → ReviewOutput
             ↓
       ReviewOutput.finalScene
             ↓
대본 출력 + 사건 저장 + 관계 갱신 + 필연 진행도 반영
             ↓
다음 장면의 get_previous_scenes Tool이 다시 조회
```

따라서 Structured Output은 단순히 JSON을 예쁘게 만드는 기능이 아니라, LLM의 결과를 프로그램이 검증하고 저장하며 다음 Agent 실행에 다시 사용할 수 있게 만드는 연결 규격입니다.

In [5]:
from pydantic import BaseModel, Field


class DialogueLine(BaseModel):
    speaker: str
    dialogue: str
    emotion: str
    action: str
    hiddenIntent: str


class Continuity(BaseModel):
    status: str = Field(pattern="^(consistent|warning)$")
    warnings: list[str]
    notes: list[str]


class FateSignal(BaseModel):
    fateId: str
    progressDelta: float = Field(ge=0, le=10)
    reason: str


class RelationshipChange(BaseModel):
    characterA: str
    characterB: str
    change: str
    reason: str


class StoryUpdates(BaseModel):
    eventSummary: str
    relationshipChanges: list[RelationshipChange]


class SceneOutput(BaseModel):
    title: str
    location: str
    stageDirection: str
    lines: list[DialogueLine] = Field(min_length=2, max_length=24)
    continuity: Continuity
    fateSignals: list[FateSignal]
    storyUpdates: StoryUpdates


class ReviewOutput(BaseModel):
    verdict: str = Field(pattern="^(approved|revised)$")
    issues: list[str]
    summary: str
    finalScene: SceneOutput

## 6. LangChain 도구
에이전트는 장면을 쓰기 전에 필요한 설정을 도구로 조회합니다. 실제 역사 기반 작품에서는 Wikipedia 검색 도구도 선택적으로 사용할 수 있습니다.

In [6]:
import requests
from langchain.tools import tool


def _json(value):
    return json.dumps(value, ensure_ascii=False)


def _characters_by_name(names):
    wanted = set(names)
    return [c for c in PROJECT["characters"] if c["name"] in wanted]


@tool
def get_character_profiles(names: list[str]) -> str:
    """캐릭터의 성격, 말투, 과거, 트라우마, 목표, 비밀과 스킬을 조회한다."""
    return _json(_characters_by_name(names))


@tool
def get_relationship_state(names: list[str]) -> str:
    """인물의 과거·현재 관계, 장면별 변화와 함께 겪은 사건을 조회한다."""
    characters = _characters_by_name(names)
    ids = {c["id"] for c in characters}
    relationships = [r for r in PROJECT["relationships"]
                     if r["characterAId"] in ids or r["characterBId"] in ids]
    events = [e for e in PROJECT["events"] if ids.intersection(e["participantIds"])]
    return _json({"characterNotes": characters, "relationships": relationships, "events": events})


@tool
def search_world_lore(query: str) -> str:
    """질문과 관련된 작품의 공식 세계관, 역사, 세력 및 규칙을 조회한다."""
    return _json({"query": query, "canon": PROJECT["world"]})


@tool
def get_inevitabilities(names: list[str]) -> str:
    """등장인물에게 반드시 일어나야 하는 필연과 성립 조건을 조회한다."""
    ids = {c["id"] for c in _characters_by_name(names)}
    return _json([f for f in PROJECT["fates"] if f["characterId"] in ids])


@tool
def get_previous_scenes(current_scene_number: int) -> str:
    """현재 장면보다 앞서 생성된 모든 장면의 사건과 대사를 시간순으로 조회한다."""
    return _json(sorted([s for s in SCENES if s["number"] < current_scene_number],
                        key=lambda scene: scene["number"]))


@tool
def search_historical_context(query: str) -> str:
    """실제 역사 기반 작품일 때 Wikipedia에서 역사적 맥락을 검색한다."""
    response = requests.get(
        "https://ko.wikipedia.org/w/api.php",
        params={"action": "query", "format": "json", "list": "search",
                "srsearch": query, "srlimit": 3},
        headers={"User-Agent": "StoryWeaver-Education/1.0"},
        timeout=10,
    )
    response.raise_for_status()
    results = response.json().get("query", {}).get("search", [])
    return _json([{"title": item["title"], "snippet": item["snippet"]} for item in results])

## 7. 두 에이전트와 Middleware 구성
작가 에이전트가 도구를 조회해 초안을 만들고, 설정 검수 에이전트가 같은 자료를 독립적으로 다시 조회해 승인하거나 교정합니다. 두 Agent에는 일시적인 모델 오류를 한 번 재시도하는 `ModelRetryMiddleware`를 적용합니다.

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware
from langchain_openai import ChatOpenAI

WRITER_PROMPT = """
당신은 영화·웹툰 작가를 돕는 StoryWeaver 장면 설계자입니다.
반드시 캐릭터 프로필, 관계 기록, 세계관 성서, 필연의 장부를 도구로 조회한 뒤 한국어 장면을 작성하세요.
장면 번호가 2 이상이면 이전 장면 도구를 반드시 호출하고 사건, 감정, 지식, 부상, 약속과 위치를 이어받으세요.
필연은 자연스러운 선택과 복선으로 조금씩 수렴시키고 매 장면에서 바로 소모하지 마세요.
기술과 초능력은 프로필에 입력된 범위와 세계관 규칙 안에서만 사용하세요. 없는 능력을 발명하지 마세요.
트라우마는 자극적으로 소비하지 말고 행동과 감각에 섬세하게 반영하세요.
long은 10~18회, standard는 6~10회의 대화 교환을 작성하세요.
이번 장면의 핵심 사건을 eventSummary에 기록하고, 실제로 달라진 관계만 relationshipChanges에 기록하세요.
설정 충돌은 continuity.warnings에 기록하세요.
"""

REVIEWER_PROMPT = """
당신은 StoryWeaver의 독립적인 설정 검수 에이전트입니다.
작가의 초안을 그대로 신뢰하지 말고 캐릭터, 관계, 세계관, 필연을 도구로 직접 다시 조회하세요.
장면 2부터는 이전 장면도 조회해 말투, 지식, 감정, 위치, 부상, 스킬 제약과 필연 조건을 검사하세요.
취향 차이는 수정하지 말고 명확한 설정 충돌만 교정하세요.
문제가 없으면 approved, 문제가 있으면 revised로 표시하고 finalScene에 최종 장면을 반환하세요.
"""

WRITER_MODEL = ChatOpenAI(
    model=os.environ.get("STORYWEAVER_MODEL", "gpt-5.4-mini"),
    temperature=0.75,
)

REVIEWER_MODEL = ChatOpenAI(
    model=os.environ.get("STORYWEAVER_REVIEW_MODEL", os.environ.get("STORYWEAVER_MODEL", "gpt-5.4-mini")),
    temperature=0.1,
)

TOOLS = [get_character_profiles, get_relationship_state, search_world_lore,
         get_inevitabilities, get_previous_scenes, search_historical_context]


def retry_middleware():
    return [ModelRetryMiddleware(
        max_retries=1, initial_delay=0.5, max_delay=2.0,
        backoff_factor=2.0, jitter=True, on_failure="error",
    )]


WRITER_AGENT = create_agent(
    model=WRITER_MODEL,
    tools=TOOLS,
    middleware=retry_middleware(),
    response_format=SceneOutput,
    system_prompt=WRITER_PROMPT,
)

REVIEWER_AGENT = create_agent(
    model=REVIEWER_MODEL,
    tools=TOOLS,
    middleware=retry_middleware(),
    response_format=ReviewOutput,
    system_prompt=REVIEWER_PROMPT,
)

print("작가 Agent와 설정 검수 Agent가 준비되었습니다.")

작가 Agent와 설정 검수 Agent가 준비되었습니다.


## 8. 장면 생성과 기억 갱신
같은 장면 번호를 다시 생성하면 해당 장면의 사건과 관계 변화를 교체하여 중복을 막습니다.

In [8]:
from IPython.display import Markdown, display


LAST_RUN_TRACE = {}


def show_tool_trace(trace):
    blocks = ["### LangChain Tool 호출"]
    for agent, names in trace.items():
        label = "작가 Agent" if agent == "writer" else "검수 Agent"
        blocks.extend(["", f"**{label}**"])
        blocks.extend(f"- `{name}`" for name in names)
    display(Markdown("\n".join(blocks)))


def show_generated_scene(review):
    scene = review.finalScene
    blocks = [
        f"## {scene.title}",
        f"**장소:** {scene.location}",
        "",
        "### 장면 설명",
        f"> {scene.stageDirection}",
        "",
        "---",
    ]
    for index, line in enumerate(scene.lines, start=1):
        blocks.extend([
            "",
            f"### {index}. {line.speaker}",
            f"> {line.dialogue}",
            "",
            f"- **감정:** {line.emotion}",
            f"- **행동:** {line.action}",
            f"- **숨은 의도:** {line.hiddenIntent}",
            "",
            "---",
        ])
    blocks.extend([
        "",
        "### 설정 검수",
        f"- **판정:** `{review.verdict}`",
        f"- **요약:** {review.summary}",
    ])
    if review.issues:
        blocks.extend(["", "**교정 사항**"])
        blocks.extend(f"- {issue}" for issue in review.issues)
    display(Markdown("\n".join(blocks)))


def _called_tools(agent_result):
    return [message.name for message in agent_result.get("messages", [])
            if getattr(message, "type", "") == "tool" and getattr(message, "name", None)]


def _character_id(name):
    character = next((c for c in PROJECT["characters"] if c["name"] == name), None)
    return character["id"] if character else None


def _apply_story_updates(scene_number, title, participants, output):
    PROJECT["events"] = [e for e in PROJECT["events"] if e["sceneNumber"] != scene_number]
    if output.storyUpdates.eventSummary.strip():
        PROJECT["events"].append({
            "id": f"event-scene-{scene_number}", "sceneNumber": scene_number,
            "title": title, "summary": output.storyUpdates.eventSummary,
            "participantIds": [cid for name in participants if (cid := _character_id(name))],
        })

    for relationship in PROJECT["relationships"]:
        relationship["history"] = [h for h in relationship["history"]
                                     if h["sceneNumber"] != scene_number]

    for change in output.storyUpdates.relationshipChanges:
        id_a, id_b = _character_id(change.characterA), _character_id(change.characterB)
        if not id_a or not id_b or id_a == id_b:
            continue
        relationship = next((r for r in PROJECT["relationships"]
                             if {r["characterAId"], r["characterBId"]} == {id_a, id_b}), None)
        history_entry = {
            "id": str(uuid4()), "sceneNumber": scene_number,
            "change": change.change, "reason": change.reason,
        }
        if relationship:
            relationship["current"] = change.change
            relationship["history"].append(history_entry)
        else:
            char_a = next(c for c in PROJECT["characters"] if c["id"] == id_a)
            char_b = next(c for c in PROJECT["characters"] if c["id"] == id_b)
            PROJECT["relationships"].append({
                "id": str(uuid4()), "characterAId": id_a, "characterBId": id_b,
                "past": " / ".join(filter(None, [char_a["relationships"], char_b["relationships"]]))
                        or "이전 관계가 설정되지 않았습니다.",
                "current": change.change, "history": [history_entry],
            })

    for signal in output.fateSignals:
        fate = next((f for f in PROJECT["fates"] if f["id"] == signal.fateId), None)
        if fate:
            fate["progress"] = min(100, fate["progress"] + signal.progressDelta)


def generate_scene(brief, participants, *, location="미정", tone="긴장감",
                   scene_number=None, dialogue_length="long"):
    global SCENES, LAST_RUN_TRACE
    if not PROJECT["characters"]:
        raise ValueError("먼저 캐릭터를 한 명 이상 추가하세요.")
    known_names = {c["name"] for c in PROJECT["characters"]}
    if not participants or not set(participants).issubset(known_names):
        raise ValueError("participants에는 등록된 캐릭터 이름만 넣으세요.")
    scene_number = scene_number or (max([s["number"] for s in SCENES], default=0) + 1)
    request_data = {
        "sceneBrief": brief, "participants": participants, "location": location,
        "tone": tone, "sceneNumber": scene_number,
        "dialogueLength": dialogue_length, "worldMode": PROJECT["world"]["mode"],
        "previousSceneCount": len([s for s in SCENES if s["number"] < scene_number]),
    }
    draft_result = WRITER_AGENT.invoke({"messages": [{"role": "user", "content": _json(request_data)}]})
    draft = draft_result["structured_response"]
    if not isinstance(draft, SceneOutput):
        draft = SceneOutput.model_validate(draft)

    review_request = {"sceneRequest": request_data, "draftScene": draft.model_dump()}
    review_result = REVIEWER_AGENT.invoke({"messages": [{"role": "user", "content": _json(review_request)}]})
    review = review_result["structured_response"]
    if not isinstance(review, ReviewOutput):
        review = ReviewOutput.model_validate(review)
    output = review.finalScene
    LAST_RUN_TRACE = {
        "writer": _called_tools(draft_result),
        "reviewer": _called_tools(review_result),
    }

    record = {"number": scene_number, "brief": brief, "participants": participants,
              **output.model_dump(),
              "agentReview": {"verdict": review.verdict, "issues": review.issues,
                              "summary": review.summary}}
    SCENES = [scene for scene in SCENES if scene["number"] != scene_number]
    SCENES.append(record)
    SCENES.sort(key=lambda scene: scene["number"])
    _apply_story_updates(scene_number, output.title, participants, output)
    return review

## 9. 시연 실행 — 장면 1
`SCENE_1_REQUEST`가 장면 요청 입력 영역입니다. 이 셀을 실행하면 작가 Agent와 검수 Agent가 차례로 동작하므로 OpenAI API 호출이 발생합니다. 연구원과 경비원은 등록하지 않은 익명 단역으로 장면 안에서만 사용합니다.

In [9]:
# ===== 발표자가 수정하는 장면 1 요청 =====
SCENE_1_REQUEST = {
    "brief": "한밤중 오스코프 연구동의 격리 장치가 폭주한다. 노먼은 사고를 유도해 피터를 시험하고, 피터는 연구원들을 구하는 과정에서 왼쪽 손목을 다친다. 노먼은 피터의 정체를 안다는 듯한 말을 남기지만 직접 폭로하지는 않는다.",
    "participants": [CHARACTER_1["name"], CHARACTER_2["name"]],
    "location": "오스코프 생명공학 연구동",
    "tone": "밀폐된 공간의 긴장감과 서로 정체를 떠보는 심리전",
    "scene_number": 1,
    "dialogue_length": "long",
}
# ===== 장면 1 요청 끝 =====

scene_1 = generate_scene(**SCENE_1_REQUEST)
display(Markdown(f"**생성된 대사 수:** {len(scene_1.finalScene.lines)}"))
show_tool_trace(LAST_RUN_TRACE)
show_generated_scene(scene_1)

**생성된 대사 수:** 17

### LangChain Tool 호출

**작가 Agent**
- `get_character_profiles`
- `get_relationship_state`
- `search_world_lore`
- `get_inevitabilities`
- `get_previous_scenes`

**검수 Agent**
- `get_character_profiles`
- `get_relationship_state`
- `search_world_lore`
- `get_inevitabilities`
- `get_previous_scenes`
- `get_previous_scenes`
- `get_relationship_state`

## 격리 장치의 밤
**장소:** 오스코프 생명공학 연구동 12층 격리 실험구역

### 장면 설명
> 한밤중. 비상등이 붉게 맥박치고, 유리 너머로 격리 챔버의 경고문이 연속으로 번쩍인다. 자동 차단문은 절반쯤 닫힌 채 삐걱거리고, 냉각 배관에서는 하얀 김이 새어 나온다. 바닥엔 깨진 샘플 카트와 쏟아진 소독액이 번들거린다. 피터는 왼손으로 벽을 짚은 채 연구원들을 출구 쪽으로 유도하고, 노먼은 통제실 쪽 어둠에서 모든 상황을 읽듯 지켜본다.

---

### 1. 노먼 오스본
> 이상하군요. 오스코프의 격리 장치는 몇 겹의 안전장치가 있는데, 오늘따라 유난히 쉽게 흔들리는군요.

- **감정:** 침착하지만 계산적인 경계
- **행동:** 통제실 창 너머로 붉은 경보를 내려다본다.
- **숨은 의도:** 사고의 반응 속도를 보며 피터의 판단을 시험한다.

---

### 2. 피터 파커
> 그 말투, 직원들 기분 나빠지라고 일부러 연습하신 거죠? 지금은 안전장치 말고 사람부터 챙겨야 할 때예요.

- **감정:** 긴장 속의 농담과 다급함
- **행동:** 깨진 바닥을 피해 연구원 한 명의 팔을 잡아 세운다.
- **숨은 의도:** 노먼의 시선을 끌어 연구원들의 대피 시간을 번다.

---

### 3. 연구원
> 격리 챔버 압력이 더 올라가고 있어요! 누군가 수동 밸브를 잠가야 해요!

- **감정:** 공포와 혼란
- **행동:** 모니터를 가리키며 뒤로 물러난다.
- **숨은 의도:** 누가 지휘권을 가져갈지 기다린다.

---

### 4. 노먼 오스본
> 수동 밸브라. 간단한 일처럼 들리지만, 간단한 일일수록 누가 정말로 유능한지 드러나지.

- **감정:** 차갑고 조롱 섞인 흥미
- **행동:** 천천히 계단을 내려와 피터를 향해 다가간다.
- **숨은 의도:** 피터가 위험을 감수하는 방식을 직접 보려 한다.

---

### 5. 피터 파커
> 좋아요. 유능함 테스트는 나중에요. 지금은 산소 호흡기가 먼저예요.

- **감정:** 조급하지만 억제된 책임감
- **행동:** 손목 시계를 확인하고, 웹 슈터를 꺼낼 자세를 취한다.
- **숨은 의도:** 노먼이 개입하기 전에 차단문을 직접 열 방법을 찾는다.

---

### 6. 노먼 오스본
> 당신은 늘 그렇죠. 위험을 보면 먼저 몸을 던집니다. 계산된 용기인지, 아니면 익숙한 죄책감인지 궁금하군요.

- **감정:** 부드러운 목소리의 압박
- **행동:** 피터의 얼굴을 유심히 관찰하며 한 발짝 옆으로 비킨다.
- **숨은 의도:** 피터의 반응에서 스파이더맨의 습관을 확인한다.

---

### 7. 피터 파커
> 와, 오스코프 인사평가 기준은 정말 살벌하네요. 죄책감 분석까지 포함이라니.

- **감정:** 겉으로는 가볍지만 속은 날 선 경계
- **행동:** 격리 챔버 쪽으로 웹을 발사해 수동 레버를 향해 몸을 날린다.
- **숨은 의도:** 연구원들을 피하게 하면서 장치의 압력 해제에 접근한다.

---

### 8. 피터 파커
> 다들 뒤로! 문 근처 말고 벽 쪽으로 붙어요!

- **감정:** 즉각적이고 단호함
- **행동:** 연구원 둘을 밀어내는 순간, 천장 패널이 번쩍이며 스파크를 튀긴다.
- **숨은 의도:** 연구원들의 동선을 고정해 추가 피해를 막는다.

---

### 9. 노먼 오스본
> 좋은 판단이군요. 그 반사신경은 단순한 실험실 재능이 아니야.

- **감정:** 낮게 깔린 확신
- **행동:** 폭주한 패널 쪽으로 시선을 돌린다.
- **숨은 의도:** 피터가 정체를 숨기고 있다는 사실을 더 확신한다.

---

### 10. 피터 파커
> 실험실 재능치곤 꽤 비싼 수업료를 치르네요!

- **감정:** 통증을 숨기려는 농담
- **행동:** 웹으로 끌어당긴 밸브 손잡이를 붙잡는 순간, 튕겨 나온 금속 파편이 왼쪽 손목을 스친다.
- **숨은 의도:** 부상을 무릅쓰고 압력 해제를 완료하려 한다.

---

### 11. 피터 파커
> 아, 좋아요. 오늘 손목은 제 편이 아니네요.

- **감정:** 짧게 밀려드는 고통
- **행동:** 이를 악물고 손목을 감싼 채, 남은 팔로 레버를 끝까지 내린다.
- **숨은 의도:** 연구원들이 더 크게 다치기 전에 격리 압력을 낮춘다.

---

### 12. 연구원
> 압력 떨어진다! 지금이에요, 출구 열렸어요!

- **감정:** 안도와 서둘러 대피
- **행동:** 비상문 쪽으로 사람들을 재촉한다.
- **숨은 의도:** 살아나갈 수 있을지 마지막으로 확인한다.

---

### 13. 노먼 오스본
> 당신은 늘 남을 먼저 살리고, 자신은 나중으로 미루는군요. 그 고집이 참... 익숙합니다.

- **감정:** 조용한 관찰과 의미심장한 확신
- **행동:** 피터의 다친 손목을 잠깐 본 뒤, 시선을 다시 얼굴로 올린다.
- **숨은 의도:** 피터가 자신이 아는 누군가와 겹친다고 떠본다.

---

### 14. 피터 파커
> 그건 칭찬으로 들으면 되나요, 아니면 오스코프식 경고로 들으면 되나요?

- **감정:** 경계하지만 웃음을 잃지 않음
- **행동:** 손목을 숨기듯 주머니에 넣고, 마지막 연구원을 출구로 밀어낸다.
- **숨은 의도:** 정체를 들키지 않은 채 노먼의 다음 말을 막아낸다.

---

### 15. 노먼 오스본
> 둘 다일 수 있죠. 다만 당신은 이미 너무 많은 책임을 한 사람 몫으로 짊어지고 있어요.

- **감정:** 차분하지만 정확히 찌르는 말투
- **행동:** 한걸음 물러서며 비상조명을 등진다.
- **숨은 의도:** 스파이더맨의 습관과 피터의 죄책감을 동시에 건드린다.

---

### 16. 피터 파커
> 책임감 얘기는 제가 제일 싫어하는 세미나 주제예요.

- **감정:** 통증을 참으며 억지로 가벼운 농담
- **행동:** 연기 속에서 기침하는 연구원을 부축해 통로 밖으로 나온다.
- **숨은 의도:** 대화를 끝내고 현장을 봉합한다.

---

### 17. 노먼 오스본
> 좋습니다. 오늘은 여기까지 하죠. 하지만 기억하세요, 피터. 당신은 생각보다 훨씬 쉽게 읽히니까.

- **감정:** 낮고 선명한 경고
- **행동:** 피터의 이름을 또렷하게 부른 뒤, 아무것도 더 말하지 않는다.
- **숨은 의도:** 정체를 폭로하지 않되, 자신이 알고 있음을 분명히 남긴다.

---

### 설정 검수
- **판정:** `approved`
- **요약:** 장면 1은 캐릭터 말투, 관계 상태, 세계관 규칙, 노먼의 정체 추궁 방식, 피터의 부상과 시민 우선 선택 모두 설정과 일치합니다.

## 10. 연속성 시연 — 장면 2
장면 2에서는 `get_previous_scenes` Tool로 장면 1을 조회합니다. 장면 1에서 다친 왼쪽 손목과 서로의 정체를 떠보던 관계가 유지되는지 결과에서 확인합니다.

In [10]:
# ===== 발표자가 수정하는 장면 2 요청 =====
SCENE_2_REQUEST = {
    "brief": "연구동 사고 직후 옥상에서 노먼은 피터에게 오스코프의 후계자가 되라고 제안한다. 피터는 장면 1에서 다친 왼쪽 손목 때문에 웹 슈터 사용이 불안정하지만 이를 숨긴다. 대화 끝에 노먼은 도시 전력망을 다음 시험 대상으로 암시한다.",
    "participants": [CHARACTER_1["name"], CHARACTER_2["name"]],
    "location": "오스코프 타워 옥상",
    "tone": "차가운 밤바람 속의 회유와 위협",
    "scene_number": 2,
    "dialogue_length": "long",
}
# ===== 장면 2 요청 끝 =====

scene_2 = generate_scene(**SCENE_2_REQUEST)
display(Markdown(f"**생성된 대사 수:** {len(scene_2.finalScene.lines)}"))
show_tool_trace(LAST_RUN_TRACE)
show_generated_scene(scene_2)

**생성된 대사 수:** 23

### LangChain Tool 호출

**작가 Agent**
- `get_character_profiles`
- `get_relationship_state`
- `search_world_lore`
- `get_inevitabilities`
- `get_previous_scenes`

**검수 Agent**
- `get_character_profiles`
- `get_relationship_state`
- `search_world_lore`
- `get_inevitabilities`
- `get_previous_scenes`

## 옥상 위의 후계자
**장소:** 오스코프 타워 옥상

### 장면 설명
> 연구동 사고가 진정된 뒤, 오스코프 타워의 옥상은 더 차갑게 느껴진다. 도시의 네온은 아래에서 희미하게 흔들리고, 강풍이 난간 사이를 파고든다. 피터는 왼쪽 손목의 통증을 억누르며 자세를 고쳐 세우고, 노먼은 비상등이 번지는 아래층을 내려다본다. 둘 사이에는 방금 전 사고의 잔향과, 아직 이름 붙이지 않은 거래가 남아 있다.

---

### 1. 노먼 오스본
> 아래는 정리됐군요. 당신이 아니었다면 더 많은 손실이 났겠지.

- **감정:** 차분한 인정과 소유욕
- **행동:** 난간에 손을 얹고 도시를 내려다본다.
- **숨은 의도:** 칭찬으로 피터의 경계를 낮춘 뒤 주도권을 쥔다.

---

### 2. 피터 파커
> 칭찬이면 고맙게 받을게요. 다만 오늘은 제 손목이 박수 치는 쪽은 아니네요.

- **감정:** 통증을 숨긴 가벼운 농담
- **행동:** 왼손을 몸 뒤로 숨기고 오른손으로만 코트를 여민다.
- **숨은 의도:** 부상을 들키지 않으려 한다.

---

### 3. 노먼 오스본
> 부상은 당신의 판단을 흐리게 만들지 못했군. 그 점이 마음에 듭니다.

- **감정:** 관찰하듯 냉정함
- **행동:** 피터의 왼쪽 팔 움직임을 짧게 훑어본다.
- **숨은 의도:** 다친 손목의 상태를 떠보고, 피터가 얼마나 버티는지 계산한다.

---

### 4. 피터 파커
> 저를 시험하시려면 차라리 퀴즈를 내세요. 옥상은 바람도 세고, 분위기도 별로예요.

- **감정:** 경계 속의 재치
- **행동:** 한 발 뒤로 물러나 바람을 정면으로 맞선다.
- **숨은 의도:** 노먼과의 물리적 거리를 유지한다.

---

### 5. 노먼 오스본
> 시험? 아니. 제안이죠. 당신 같은 재능이 작은 실험실에 묶여 있을 이유는 없습니다.

- **감정:** 회유하는 듯한 권위
- **행동:** 주머니에서 장갑 낀 손을 빼며 천천히 피터 쪽으로 몸을 돌린다.
- **숨은 의도:** 피터를 오스코프의 통제 안으로 끌어들이려 한다.

---

### 6. 피터 파커
> 재능이요? 오늘은 그 재능이 사람들 살리는 데만 쓰였으면 좋겠네요.

- **감정:** 단호하지만 억제된 불신
- **행동:** 시선을 피하지 않고 노먼을 똑바로 본다.
- **숨은 의도:** 후계자 제안을 거절할 빌미를 찾는다.

---

### 7. 노먼 오스본
> 바로 그겁니다. 사람을 살릴 수 있는 사람은, 결국 많은 것을 바꿀 수 있죠. 오스코프는 그런 사람을 다음 자리에 앉혀야 합니다.

- **감정:** 조용한 설득
- **행동:** 도시를 가리키듯 턱으로 아래층을 가리킨다.
- **숨은 의도:** ‘후계자’라는 말을 직접 꺼내 피터의 반응을 읽는다.

---

### 8. 피터 파커
> 설마 지금, 오스코프 후계자 얘기하시는 건가요? 저는 경영 수업도 안 들었는데요.

- **감정:** 놀람을 감춘 농담
- **행동:** 짧게 웃는 척하지만 왼손의 긴장을 풀지 못한다.
- **숨은 의도:** 제안의 진심과 함정을 동시에 확인한다.

---

### 9. 노먼 오스본
> 경영 수업은 숫자를 읽는 법을 가르칠 뿐입니다. 진짜 중요한 건, 필요한 순간에 누구를 남기고 누구를 데려갈지 아는 감각이죠.

- **감정:** 차갑고 교육적인 어조
- **행동:** 비상구 쪽에서 새어 나오는 붉은 불빛을 등진다.
- **숨은 의도:** 피터의 책임감과 죄책감을 건드려 동의하게 만들려 한다.

---

### 10. 피터 파커
> 그 말, 들을수록 영업보단 협박 쪽에 가까운데요.

- **감정:** 예민한 경계
- **행동:** 재킷 소매를 올렸다가, 왼쪽 손목 통증 때문에 바로 내린다.
- **숨은 의도:** 부상을 감추려다 순간적으로 반응이 새나온다.

---

### 11. 노먼 오스본
> 협박이라면 더 직접적이었겠죠. 저는 당신에게 가능성을 주려는 겁니다. 스파이더맨이든, 피터 파커든.

- **감정:** 부드럽지만 날카로운 확신
- **행동:** 피터의 이름과 별칭을 나란히 놓듯 천천히 발음한다.
- **숨은 의도:** 정체를 알고 있다는 압박을 다시 건드린다.

---

### 12. 피터 파커
> 그 둘을 같은 문장에 넣는 건 좀 위험한 취미네요.

- **감정:** 순간 얼어붙은 경계
- **행동:** 어깨를 굳힌 채 시선을 잠깐 옆으로 피한다.
- **숨은 의도:** 정체 노출을 피하며 대화를 넘긴다.

---

### 13. 노먼 오스본
> 당신은 이미 위험한 사람입니다. 그걸 숨긴 채 일하는 건 비효율적이죠.

- **감정:** 낮게 깔린 확신
- **행동:** 작은 미소를 보이다가 곧바로 사라지게 한다.
- **숨은 의도:** 피터가 자신을 거절하기 어려운 이유를 파악한다.

---

### 14. 피터 파커
> 저는 비효율보다 과열을 더 싫어해요. 오늘 연구동도 그랬고요.

- **감정:** 정중하지만 차가운 거절
- **행동:** 손목을 완전히 숨기기 위해 다른 손으로 팔을 감싼다.
- **숨은 의도:** 연구동 사고를 언급해 노먼의 책임을 간접적으로 찌른다.

---

### 15. 노먼 오스본
> 아, 그 사고요. 당신은 아직도 그게 단순한 사고였다고 믿습니까?

- **감정:** 침착한 위협
- **행동:** 고개를 약간 기울여 피터의 표정을 읽는다.
- **숨은 의도:** 사고의 의도를 자각하게 만들어 심리적 우위를 잡는다.

---

### 16. 피터 파커
> 노먼, 지금 그 질문은 대답보다 변명이 더 필요해 보이는데요.

- **감정:** 억눌린 분노를 농담으로 감춤
- **행동:** 한 걸음 다가가다가 바람에 흔들리는 균형을 바로잡는다.
- **숨은 의도:** 더 파고들면 자신도 흔들릴 수 있음을 경계한다.

---

### 17. 노먼 오스본
> 좋습니다. 그럼 다른 질문을 하죠. 뉴욕의 전력망이 얼마나 취약한지, 당신은 관심 있어요?

- **감정:** 조용히 던지는 유혹
- **행동:** 도시 불빛이 모여 있는 방향을 손가락 끝으로 가리킨다.
- **숨은 의도:** 다음 시험 대상이 도시 전력망임을 암시하며 피터를 끌어들이려 한다.

---

### 18. 피터 파커
> 전력망요? 갑자기 도시 전체를 시험대로 쓰자는 건가요.

- **감정:** 경계가 확실해진 목소리
- **행동:** 왼손의 미세한 떨림을 숨기려 손가락을 주먹 쥔다.
- **숨은 의도:** 구체적 계획을 캐내려 하지만 손목 상태 때문에 여유가 줄어든다.

---

### 19. 노먼 오스본
> 시험은 늘 필요한 겁니다. 약한 지점이 드러나야 고칠 수 있으니까. 뉴욕은 생각보다 많은 것을 저에게 빚지고 있죠.

- **감정:** 권력자의 확신
- **행동:** 코트 자락을 바람에 맡긴 채 옥상 끝으로 한 발 옮긴다.
- **숨은 의도:** 전력망 실험이 오스코프의 다음 단계임을 기정사실처럼 흘린다.

---

### 20. 피터 파커
> 도시를 고치는 방법이 꼭 도시를 흔드는 일이어야 하진 않죠.

- **감정:** 단호함과 피로
- **행동:** 짧게 숨을 들이켜며 통증을 견디고, 노먼의 동선을 눈으로 따라간다.
- **숨은 의도:** 노먼을 말로 묶어 더 이상의 구체적 실행을 막으려 한다.

---

### 21. 노먼 오스본
> 당신답군요. 늘 결과보다 사람을 먼저 보니까.

- **감정:** 조용한 평가
- **행동:** 피터를 한 번 더 훑어본 뒤, 의미심장하게 미소 짓는다.
- **숨은 의도:** 피터가 전력망 문제에 개입할 가능성을 계산한다.

---

### 22. 피터 파커
> 그게 제 결함처럼 들리네요.

- **감정:** 쓴웃음
- **행동:** 오른손으로 난간을 짚으며 균형을 잡는다.
- **숨은 의도:** 대화를 끝내기 전에 더 불리한 약속을 만들지 않으려 한다.

---

### 23. 노먼 오스본
> 결함이 아니라, 선택이죠. 그리고 선택은 곧 책임입니다. 다음에 만날 때는 당신이 어느 쪽에 설지 더 분명해져 있길 바랍니다, 피터.

- **감정:** 차갑게 정리하는 어조
- **행동:** 뒤돌아 서서 옥상 출입문 쪽으로 걸음을 옮긴다.
- **숨은 의도:** 후계자 제안을 미끼로 남겨두고, 전력망 계획에 대한 피터의 반응을 숙제로 만든다.

---

### 설정 검수
- **판정:** `approved`
- **요약:** 장면 2는 장면 1의 부상, 관계 악화, 노먼의 심리전과 비밀 정체 암시를 자연스럽게 이어받았고, 피터의 말투와 세계관 규칙에도 충돌이 없습니다. 노먼이 오스코프 후계자 제안과 도시 전력망 시험을 연결하는 흐름도 설정상 타당합니다.

## 11. 결과 확인과 최종 대본

In [11]:
from IPython.display import Markdown, display


def show_scene(scene_number):
    scene = next(s for s in SCENES if s["number"] == scene_number)
    text = [f"## 장면 {scene_number}. {scene['title']}",
            f"**장소:** {scene['location']}", "", f"*{scene['stageDirection']}*", ""]
    for line in scene["lines"]:
        text.extend([f"**{line['speaker']}** · {line['emotion']}",
                     f"> {line['dialogue']}", f"*{line['action']}*", ""])
    display(Markdown("\n".join(text)))


def show_relationships_and_events():
    names = {c["id"]: c["name"] for c in PROJECT["characters"]}
    text = ["## 관계 변화"]
    for relation in PROJECT["relationships"]:
        text.extend([f"### {names.get(relation['characterAId'], '?')} ↔ {names.get(relation['characterBId'], '?')}",
                     f"- 과거: {relation['past']}", f"- 현재: {relation['current']}"])
        for history in sorted(relation["history"], key=lambda item: item["sceneNumber"]):
            text.append(f"  - 장면 {history['sceneNumber']}: {history['change']} — {history['reason']}")
    text.append("\n## 사건 연대기")
    for event in sorted(PROJECT["events"], key=lambda item: item["sceneNumber"]):
        text.append(f"- **장면 {event['sceneNumber']} · {event['title']}**: {event['summary']}")
    display(Markdown("\n".join(text)))


def final_script():
    sections = [f"# {PROJECT['world']['title']}", "## 최종 대본"]
    for scene in sorted(SCENES, key=lambda item: item["number"]):
        sections.extend([f"\n### 장면 {scene['number']}. {scene['title']}",
                         f"장소: {scene['location']}", f"[{scene['stageDirection']}]"])
        for line in scene["lines"]:
            sections.extend([f"\n{line['speaker']} ({line['emotion']})",
                             line["dialogue"], f"[{line['action']}]"])
    return "\n".join(sections)


show_scene(1)
show_scene(2)
show_relationships_and_events()
display(Markdown(final_script()))

## 장면 1. 격리 장치의 밤
**장소:** 오스코프 생명공학 연구동 12층 격리 실험구역

*한밤중. 비상등이 붉게 맥박치고, 유리 너머로 격리 챔버의 경고문이 연속으로 번쩍인다. 자동 차단문은 절반쯤 닫힌 채 삐걱거리고, 냉각 배관에서는 하얀 김이 새어 나온다. 바닥엔 깨진 샘플 카트와 쏟아진 소독액이 번들거린다. 피터는 왼손으로 벽을 짚은 채 연구원들을 출구 쪽으로 유도하고, 노먼은 통제실 쪽 어둠에서 모든 상황을 읽듯 지켜본다.*

**노먼 오스본** · 침착하지만 계산적인 경계
> 이상하군요. 오스코프의 격리 장치는 몇 겹의 안전장치가 있는데, 오늘따라 유난히 쉽게 흔들리는군요.
*통제실 창 너머로 붉은 경보를 내려다본다.*

**피터 파커** · 긴장 속의 농담과 다급함
> 그 말투, 직원들 기분 나빠지라고 일부러 연습하신 거죠? 지금은 안전장치 말고 사람부터 챙겨야 할 때예요.
*깨진 바닥을 피해 연구원 한 명의 팔을 잡아 세운다.*

**연구원** · 공포와 혼란
> 격리 챔버 압력이 더 올라가고 있어요! 누군가 수동 밸브를 잠가야 해요!
*모니터를 가리키며 뒤로 물러난다.*

**노먼 오스본** · 차갑고 조롱 섞인 흥미
> 수동 밸브라. 간단한 일처럼 들리지만, 간단한 일일수록 누가 정말로 유능한지 드러나지.
*천천히 계단을 내려와 피터를 향해 다가간다.*

**피터 파커** · 조급하지만 억제된 책임감
> 좋아요. 유능함 테스트는 나중에요. 지금은 산소 호흡기가 먼저예요.
*손목 시계를 확인하고, 웹 슈터를 꺼낼 자세를 취한다.*

**노먼 오스본** · 부드러운 목소리의 압박
> 당신은 늘 그렇죠. 위험을 보면 먼저 몸을 던집니다. 계산된 용기인지, 아니면 익숙한 죄책감인지 궁금하군요.
*피터의 얼굴을 유심히 관찰하며 한 발짝 옆으로 비킨다.*

**피터 파커** · 겉으로는 가볍지만 속은 날 선 경계
> 와, 오스코프 인사평가 기준은 정말 살벌하네요. 죄책감 분석까지 포함이라니.
*격리 챔버 쪽으로 웹을 발사해 수동 레버를 향해 몸을 날린다.*

**피터 파커** · 즉각적이고 단호함
> 다들 뒤로! 문 근처 말고 벽 쪽으로 붙어요!
*연구원 둘을 밀어내는 순간, 천장 패널이 번쩍이며 스파크를 튀긴다.*

**노먼 오스본** · 낮게 깔린 확신
> 좋은 판단이군요. 그 반사신경은 단순한 실험실 재능이 아니야.
*폭주한 패널 쪽으로 시선을 돌린다.*

**피터 파커** · 통증을 숨기려는 농담
> 실험실 재능치곤 꽤 비싼 수업료를 치르네요!
*웹으로 끌어당긴 밸브 손잡이를 붙잡는 순간, 튕겨 나온 금속 파편이 왼쪽 손목을 스친다.*

**피터 파커** · 짧게 밀려드는 고통
> 아, 좋아요. 오늘 손목은 제 편이 아니네요.
*이를 악물고 손목을 감싼 채, 남은 팔로 레버를 끝까지 내린다.*

**연구원** · 안도와 서둘러 대피
> 압력 떨어진다! 지금이에요, 출구 열렸어요!
*비상문 쪽으로 사람들을 재촉한다.*

**노먼 오스본** · 조용한 관찰과 의미심장한 확신
> 당신은 늘 남을 먼저 살리고, 자신은 나중으로 미루는군요. 그 고집이 참... 익숙합니다.
*피터의 다친 손목을 잠깐 본 뒤, 시선을 다시 얼굴로 올린다.*

**피터 파커** · 경계하지만 웃음을 잃지 않음
> 그건 칭찬으로 들으면 되나요, 아니면 오스코프식 경고로 들으면 되나요?
*손목을 숨기듯 주머니에 넣고, 마지막 연구원을 출구로 밀어낸다.*

**노먼 오스본** · 차분하지만 정확히 찌르는 말투
> 둘 다일 수 있죠. 다만 당신은 이미 너무 많은 책임을 한 사람 몫으로 짊어지고 있어요.
*한걸음 물러서며 비상조명을 등진다.*

**피터 파커** · 통증을 참으며 억지로 가벼운 농담
> 책임감 얘기는 제가 제일 싫어하는 세미나 주제예요.
*연기 속에서 기침하는 연구원을 부축해 통로 밖으로 나온다.*

**노먼 오스본** · 낮고 선명한 경고
> 좋습니다. 오늘은 여기까지 하죠. 하지만 기억하세요, 피터. 당신은 생각보다 훨씬 쉽게 읽히니까.
*피터의 이름을 또렷하게 부른 뒤, 아무것도 더 말하지 않는다.*


## 장면 2. 옥상 위의 후계자
**장소:** 오스코프 타워 옥상

*연구동 사고가 진정된 뒤, 오스코프 타워의 옥상은 더 차갑게 느껴진다. 도시의 네온은 아래에서 희미하게 흔들리고, 강풍이 난간 사이를 파고든다. 피터는 왼쪽 손목의 통증을 억누르며 자세를 고쳐 세우고, 노먼은 비상등이 번지는 아래층을 내려다본다. 둘 사이에는 방금 전 사고의 잔향과, 아직 이름 붙이지 않은 거래가 남아 있다.*

**노먼 오스본** · 차분한 인정과 소유욕
> 아래는 정리됐군요. 당신이 아니었다면 더 많은 손실이 났겠지.
*난간에 손을 얹고 도시를 내려다본다.*

**피터 파커** · 통증을 숨긴 가벼운 농담
> 칭찬이면 고맙게 받을게요. 다만 오늘은 제 손목이 박수 치는 쪽은 아니네요.
*왼손을 몸 뒤로 숨기고 오른손으로만 코트를 여민다.*

**노먼 오스본** · 관찰하듯 냉정함
> 부상은 당신의 판단을 흐리게 만들지 못했군. 그 점이 마음에 듭니다.
*피터의 왼쪽 팔 움직임을 짧게 훑어본다.*

**피터 파커** · 경계 속의 재치
> 저를 시험하시려면 차라리 퀴즈를 내세요. 옥상은 바람도 세고, 분위기도 별로예요.
*한 발 뒤로 물러나 바람을 정면으로 맞선다.*

**노먼 오스본** · 회유하는 듯한 권위
> 시험? 아니. 제안이죠. 당신 같은 재능이 작은 실험실에 묶여 있을 이유는 없습니다.
*주머니에서 장갑 낀 손을 빼며 천천히 피터 쪽으로 몸을 돌린다.*

**피터 파커** · 단호하지만 억제된 불신
> 재능이요? 오늘은 그 재능이 사람들 살리는 데만 쓰였으면 좋겠네요.
*시선을 피하지 않고 노먼을 똑바로 본다.*

**노먼 오스본** · 조용한 설득
> 바로 그겁니다. 사람을 살릴 수 있는 사람은, 결국 많은 것을 바꿀 수 있죠. 오스코프는 그런 사람을 다음 자리에 앉혀야 합니다.
*도시를 가리키듯 턱으로 아래층을 가리킨다.*

**피터 파커** · 놀람을 감춘 농담
> 설마 지금, 오스코프 후계자 얘기하시는 건가요? 저는 경영 수업도 안 들었는데요.
*짧게 웃는 척하지만 왼손의 긴장을 풀지 못한다.*

**노먼 오스본** · 차갑고 교육적인 어조
> 경영 수업은 숫자를 읽는 법을 가르칠 뿐입니다. 진짜 중요한 건, 필요한 순간에 누구를 남기고 누구를 데려갈지 아는 감각이죠.
*비상구 쪽에서 새어 나오는 붉은 불빛을 등진다.*

**피터 파커** · 예민한 경계
> 그 말, 들을수록 영업보단 협박 쪽에 가까운데요.
*재킷 소매를 올렸다가, 왼쪽 손목 통증 때문에 바로 내린다.*

**노먼 오스본** · 부드럽지만 날카로운 확신
> 협박이라면 더 직접적이었겠죠. 저는 당신에게 가능성을 주려는 겁니다. 스파이더맨이든, 피터 파커든.
*피터의 이름과 별칭을 나란히 놓듯 천천히 발음한다.*

**피터 파커** · 순간 얼어붙은 경계
> 그 둘을 같은 문장에 넣는 건 좀 위험한 취미네요.
*어깨를 굳힌 채 시선을 잠깐 옆으로 피한다.*

**노먼 오스본** · 낮게 깔린 확신
> 당신은 이미 위험한 사람입니다. 그걸 숨긴 채 일하는 건 비효율적이죠.
*작은 미소를 보이다가 곧바로 사라지게 한다.*

**피터 파커** · 정중하지만 차가운 거절
> 저는 비효율보다 과열을 더 싫어해요. 오늘 연구동도 그랬고요.
*손목을 완전히 숨기기 위해 다른 손으로 팔을 감싼다.*

**노먼 오스본** · 침착한 위협
> 아, 그 사고요. 당신은 아직도 그게 단순한 사고였다고 믿습니까?
*고개를 약간 기울여 피터의 표정을 읽는다.*

**피터 파커** · 억눌린 분노를 농담으로 감춤
> 노먼, 지금 그 질문은 대답보다 변명이 더 필요해 보이는데요.
*한 걸음 다가가다가 바람에 흔들리는 균형을 바로잡는다.*

**노먼 오스본** · 조용히 던지는 유혹
> 좋습니다. 그럼 다른 질문을 하죠. 뉴욕의 전력망이 얼마나 취약한지, 당신은 관심 있어요?
*도시 불빛이 모여 있는 방향을 손가락 끝으로 가리킨다.*

**피터 파커** · 경계가 확실해진 목소리
> 전력망요? 갑자기 도시 전체를 시험대로 쓰자는 건가요.
*왼손의 미세한 떨림을 숨기려 손가락을 주먹 쥔다.*

**노먼 오스본** · 권력자의 확신
> 시험은 늘 필요한 겁니다. 약한 지점이 드러나야 고칠 수 있으니까. 뉴욕은 생각보다 많은 것을 저에게 빚지고 있죠.
*코트 자락을 바람에 맡긴 채 옥상 끝으로 한 발 옮긴다.*

**피터 파커** · 단호함과 피로
> 도시를 고치는 방법이 꼭 도시를 흔드는 일이어야 하진 않죠.
*짧게 숨을 들이켜며 통증을 견디고, 노먼의 동선을 눈으로 따라간다.*

**노먼 오스본** · 조용한 평가
> 당신답군요. 늘 결과보다 사람을 먼저 보니까.
*피터를 한 번 더 훑어본 뒤, 의미심장하게 미소 짓는다.*

**피터 파커** · 쓴웃음
> 그게 제 결함처럼 들리네요.
*오른손으로 난간을 짚으며 균형을 잡는다.*

**노먼 오스본** · 차갑게 정리하는 어조
> 결함이 아니라, 선택이죠. 그리고 선택은 곧 책임입니다. 다음에 만날 때는 당신이 어느 쪽에 설지 더 분명해져 있길 바랍니다, 피터.
*뒤돌아 서서 옥상 출입문 쪽으로 걸음을 옮긴다.*


## 관계 변화
### 피터 파커 ↔ 노먼 오스본
- 과거: 노먼은 피터의 과학적 재능을 높이 평가했고 피터는 노먼의 연구가 가진 위험성을 목격했다.
- 현재: 회유와 위협이 결합된 심리전으로 불신이 한층 심화됨
  - 장면 1: 더 깊은 경계와 불신이 형성됨 — 노먼이 피터의 이름과 죄책감을 정확히 찌르며 정체를 아는 듯한 암시를 남겼기 때문
  - 장면 2: 회유와 위협이 결합된 심리전으로 불신이 한층 심화됨 — 노먼이 후계자 제안과 정체 암시, 전력망 계획을 함께 내밀어 피터를 압박했기 때문

## 사건 연대기
- **장면 1 · 격리 장치의 밤**: 한밤중 오스코프 연구동의 격리 장치가 폭주했으며, 피터가 연구원들을 대피시키고 수동 압력 해제를 성공시켰다. 그 과정에서 왼쪽 손목을 다쳤고, 노먼은 피터의 정체를 눈치챈 듯한 말로 심리를 압박했다.
- **장면 2 · 옥상 위의 후계자**: 오스코프 타워 옥상에서 노먼이 피터에게 오스코프 후계자 자리를 제안하며 회유했고, 피터는 왼쪽 손목 부상을 숨긴 채 이를 경계했다. 대화 끝에 노먼은 도시 전력망을 다음 시험 대상으로 암시하며 피터를 압박했다.

# 거미줄 너머의 선택
## 최종 대본

### 장면 1. 격리 장치의 밤
장소: 오스코프 생명공학 연구동 12층 격리 실험구역
[한밤중. 비상등이 붉게 맥박치고, 유리 너머로 격리 챔버의 경고문이 연속으로 번쩍인다. 자동 차단문은 절반쯤 닫힌 채 삐걱거리고, 냉각 배관에서는 하얀 김이 새어 나온다. 바닥엔 깨진 샘플 카트와 쏟아진 소독액이 번들거린다. 피터는 왼손으로 벽을 짚은 채 연구원들을 출구 쪽으로 유도하고, 노먼은 통제실 쪽 어둠에서 모든 상황을 읽듯 지켜본다.]

노먼 오스본 (침착하지만 계산적인 경계)
이상하군요. 오스코프의 격리 장치는 몇 겹의 안전장치가 있는데, 오늘따라 유난히 쉽게 흔들리는군요.
[통제실 창 너머로 붉은 경보를 내려다본다.]

피터 파커 (긴장 속의 농담과 다급함)
그 말투, 직원들 기분 나빠지라고 일부러 연습하신 거죠? 지금은 안전장치 말고 사람부터 챙겨야 할 때예요.
[깨진 바닥을 피해 연구원 한 명의 팔을 잡아 세운다.]

연구원 (공포와 혼란)
격리 챔버 압력이 더 올라가고 있어요! 누군가 수동 밸브를 잠가야 해요!
[모니터를 가리키며 뒤로 물러난다.]

노먼 오스본 (차갑고 조롱 섞인 흥미)
수동 밸브라. 간단한 일처럼 들리지만, 간단한 일일수록 누가 정말로 유능한지 드러나지.
[천천히 계단을 내려와 피터를 향해 다가간다.]

피터 파커 (조급하지만 억제된 책임감)
좋아요. 유능함 테스트는 나중에요. 지금은 산소 호흡기가 먼저예요.
[손목 시계를 확인하고, 웹 슈터를 꺼낼 자세를 취한다.]

노먼 오스본 (부드러운 목소리의 압박)
당신은 늘 그렇죠. 위험을 보면 먼저 몸을 던집니다. 계산된 용기인지, 아니면 익숙한 죄책감인지 궁금하군요.
[피터의 얼굴을 유심히 관찰하며 한 발짝 옆으로 비킨다.]

피터 파커 (겉으로는 가볍지만 속은 날 선 경계)
와, 오스코프 인사평가 기준은 정말 살벌하네요. 죄책감 분석까지 포함이라니.
[격리 챔버 쪽으로 웹을 발사해 수동 레버를 향해 몸을 날린다.]

피터 파커 (즉각적이고 단호함)
다들 뒤로! 문 근처 말고 벽 쪽으로 붙어요!
[연구원 둘을 밀어내는 순간, 천장 패널이 번쩍이며 스파크를 튀긴다.]

노먼 오스본 (낮게 깔린 확신)
좋은 판단이군요. 그 반사신경은 단순한 실험실 재능이 아니야.
[폭주한 패널 쪽으로 시선을 돌린다.]

피터 파커 (통증을 숨기려는 농담)
실험실 재능치곤 꽤 비싼 수업료를 치르네요!
[웹으로 끌어당긴 밸브 손잡이를 붙잡는 순간, 튕겨 나온 금속 파편이 왼쪽 손목을 스친다.]

피터 파커 (짧게 밀려드는 고통)
아, 좋아요. 오늘 손목은 제 편이 아니네요.
[이를 악물고 손목을 감싼 채, 남은 팔로 레버를 끝까지 내린다.]

연구원 (안도와 서둘러 대피)
압력 떨어진다! 지금이에요, 출구 열렸어요!
[비상문 쪽으로 사람들을 재촉한다.]

노먼 오스본 (조용한 관찰과 의미심장한 확신)
당신은 늘 남을 먼저 살리고, 자신은 나중으로 미루는군요. 그 고집이 참... 익숙합니다.
[피터의 다친 손목을 잠깐 본 뒤, 시선을 다시 얼굴로 올린다.]

피터 파커 (경계하지만 웃음을 잃지 않음)
그건 칭찬으로 들으면 되나요, 아니면 오스코프식 경고로 들으면 되나요?
[손목을 숨기듯 주머니에 넣고, 마지막 연구원을 출구로 밀어낸다.]

노먼 오스본 (차분하지만 정확히 찌르는 말투)
둘 다일 수 있죠. 다만 당신은 이미 너무 많은 책임을 한 사람 몫으로 짊어지고 있어요.
[한걸음 물러서며 비상조명을 등진다.]

피터 파커 (통증을 참으며 억지로 가벼운 농담)
책임감 얘기는 제가 제일 싫어하는 세미나 주제예요.
[연기 속에서 기침하는 연구원을 부축해 통로 밖으로 나온다.]

노먼 오스본 (낮고 선명한 경고)
좋습니다. 오늘은 여기까지 하죠. 하지만 기억하세요, 피터. 당신은 생각보다 훨씬 쉽게 읽히니까.
[피터의 이름을 또렷하게 부른 뒤, 아무것도 더 말하지 않는다.]

### 장면 2. 옥상 위의 후계자
장소: 오스코프 타워 옥상
[연구동 사고가 진정된 뒤, 오스코프 타워의 옥상은 더 차갑게 느껴진다. 도시의 네온은 아래에서 희미하게 흔들리고, 강풍이 난간 사이를 파고든다. 피터는 왼쪽 손목의 통증을 억누르며 자세를 고쳐 세우고, 노먼은 비상등이 번지는 아래층을 내려다본다. 둘 사이에는 방금 전 사고의 잔향과, 아직 이름 붙이지 않은 거래가 남아 있다.]

노먼 오스본 (차분한 인정과 소유욕)
아래는 정리됐군요. 당신이 아니었다면 더 많은 손실이 났겠지.
[난간에 손을 얹고 도시를 내려다본다.]

피터 파커 (통증을 숨긴 가벼운 농담)
칭찬이면 고맙게 받을게요. 다만 오늘은 제 손목이 박수 치는 쪽은 아니네요.
[왼손을 몸 뒤로 숨기고 오른손으로만 코트를 여민다.]

노먼 오스본 (관찰하듯 냉정함)
부상은 당신의 판단을 흐리게 만들지 못했군. 그 점이 마음에 듭니다.
[피터의 왼쪽 팔 움직임을 짧게 훑어본다.]

피터 파커 (경계 속의 재치)
저를 시험하시려면 차라리 퀴즈를 내세요. 옥상은 바람도 세고, 분위기도 별로예요.
[한 발 뒤로 물러나 바람을 정면으로 맞선다.]

노먼 오스본 (회유하는 듯한 권위)
시험? 아니. 제안이죠. 당신 같은 재능이 작은 실험실에 묶여 있을 이유는 없습니다.
[주머니에서 장갑 낀 손을 빼며 천천히 피터 쪽으로 몸을 돌린다.]

피터 파커 (단호하지만 억제된 불신)
재능이요? 오늘은 그 재능이 사람들 살리는 데만 쓰였으면 좋겠네요.
[시선을 피하지 않고 노먼을 똑바로 본다.]

노먼 오스본 (조용한 설득)
바로 그겁니다. 사람을 살릴 수 있는 사람은, 결국 많은 것을 바꿀 수 있죠. 오스코프는 그런 사람을 다음 자리에 앉혀야 합니다.
[도시를 가리키듯 턱으로 아래층을 가리킨다.]

피터 파커 (놀람을 감춘 농담)
설마 지금, 오스코프 후계자 얘기하시는 건가요? 저는 경영 수업도 안 들었는데요.
[짧게 웃는 척하지만 왼손의 긴장을 풀지 못한다.]

노먼 오스본 (차갑고 교육적인 어조)
경영 수업은 숫자를 읽는 법을 가르칠 뿐입니다. 진짜 중요한 건, 필요한 순간에 누구를 남기고 누구를 데려갈지 아는 감각이죠.
[비상구 쪽에서 새어 나오는 붉은 불빛을 등진다.]

피터 파커 (예민한 경계)
그 말, 들을수록 영업보단 협박 쪽에 가까운데요.
[재킷 소매를 올렸다가, 왼쪽 손목 통증 때문에 바로 내린다.]

노먼 오스본 (부드럽지만 날카로운 확신)
협박이라면 더 직접적이었겠죠. 저는 당신에게 가능성을 주려는 겁니다. 스파이더맨이든, 피터 파커든.
[피터의 이름과 별칭을 나란히 놓듯 천천히 발음한다.]

피터 파커 (순간 얼어붙은 경계)
그 둘을 같은 문장에 넣는 건 좀 위험한 취미네요.
[어깨를 굳힌 채 시선을 잠깐 옆으로 피한다.]

노먼 오스본 (낮게 깔린 확신)
당신은 이미 위험한 사람입니다. 그걸 숨긴 채 일하는 건 비효율적이죠.
[작은 미소를 보이다가 곧바로 사라지게 한다.]

피터 파커 (정중하지만 차가운 거절)
저는 비효율보다 과열을 더 싫어해요. 오늘 연구동도 그랬고요.
[손목을 완전히 숨기기 위해 다른 손으로 팔을 감싼다.]

노먼 오스본 (침착한 위협)
아, 그 사고요. 당신은 아직도 그게 단순한 사고였다고 믿습니까?
[고개를 약간 기울여 피터의 표정을 읽는다.]

피터 파커 (억눌린 분노를 농담으로 감춤)
노먼, 지금 그 질문은 대답보다 변명이 더 필요해 보이는데요.
[한 걸음 다가가다가 바람에 흔들리는 균형을 바로잡는다.]

노먼 오스본 (조용히 던지는 유혹)
좋습니다. 그럼 다른 질문을 하죠. 뉴욕의 전력망이 얼마나 취약한지, 당신은 관심 있어요?
[도시 불빛이 모여 있는 방향을 손가락 끝으로 가리킨다.]

피터 파커 (경계가 확실해진 목소리)
전력망요? 갑자기 도시 전체를 시험대로 쓰자는 건가요.
[왼손의 미세한 떨림을 숨기려 손가락을 주먹 쥔다.]

노먼 오스본 (권력자의 확신)
시험은 늘 필요한 겁니다. 약한 지점이 드러나야 고칠 수 있으니까. 뉴욕은 생각보다 많은 것을 저에게 빚지고 있죠.
[코트 자락을 바람에 맡긴 채 옥상 끝으로 한 발 옮긴다.]

피터 파커 (단호함과 피로)
도시를 고치는 방법이 꼭 도시를 흔드는 일이어야 하진 않죠.
[짧게 숨을 들이켜며 통증을 견디고, 노먼의 동선을 눈으로 따라간다.]

노먼 오스본 (조용한 평가)
당신답군요. 늘 결과보다 사람을 먼저 보니까.
[피터를 한 번 더 훑어본 뒤, 의미심장하게 미소 짓는다.]

피터 파커 (쓴웃음)
그게 제 결함처럼 들리네요.
[오른손으로 난간을 짚으며 균형을 잡는다.]

노먼 오스본 (차갑게 정리하는 어조)
결함이 아니라, 선택이죠. 그리고 선택은 곧 책임입니다. 다음에 만날 때는 당신이 어느 쪽에 설지 더 분명해져 있길 바랍니다, 피터.
[뒤돌아 서서 옥상 출입문 쪽으로 걸음을 옮긴다.]

## 설계 선택의 이유

### 1. RAG가 아니라 Tool을 사용한 이유
StoryWeaver의 핵심 데이터는 캐릭터 이름, 캐릭터 ID, 관계 ID, 필연 ID, 장면 번호처럼 구조가 명확합니다. 예를 들어 피터 파커의 설정을 찾을 때는 의미가 비슷한 문서를 찾는 것보다 `피터 파커`라는 정확한 이름에 연결된 전체 프로필을 가져오는 것이 중요합니다.

Vector Store 기반 RAG는 긴 문서 중 질문과 의미가 비슷한 일부 내용을 찾는 데 유용하지만, 유사도에 따라 필요한 설정이 누락되거나 다른 인물의 설정이 함께 검색될 수 있습니다. 반면 Tool은 어떤 데이터에서 무엇을 조회할지 명시할 수 있고, Agent가 호출한 Tool과 결과를 추적할 수 있습니다. 따라서 현재처럼 데이터가 구조화되어 있고 정확한 조회가 필요한 프로젝트에는 Tool 방식이 더 적합합니다.

- `get_character_profiles`: 이름으로 캐릭터 전체 설정을 정확하게 조회합니다.
- `get_relationship_state`: 캐릭터 ID로 관계와 사건을 조회합니다.
- `get_inevitabilities`: 캐릭터에게 연결된 필연 ID와 조건을 조회합니다.
- `get_previous_scenes`: 장면 번호를 기준으로 이전 기록을 시간순으로 조회합니다.
- `search_historical_context`: 외부 정보가 필요할 때만 Wikipedia API를 호출합니다.

향후 사용자가 장편 설정집, 소설 원문, 역사 자료처럼 구조화되지 않은 대용량 문서를 업로드하는 기능을 추가한다면 그 문서 검색에는 Embedding과 Retriever 기반 RAG를 함께 사용할 수 있습니다. 즉, 현재 구조에서 Tool과 RAG는 경쟁 관계가 아니라 데이터 형태에 따라 선택하는 방식입니다.

### 2. 결과를 Structured Output으로 저장한 이유
LLM이 자유 형식 텍스트만 반환하면 프로그램은 어디까지가 대사이고, 무엇이 행동·감정·관계 변화인지 안정적으로 구분할 수 없습니다. StoryWeaver는 `SceneOutput`과 `ReviewOutput`을 `response_format`으로 지정하여 결과를 정해진 필드와 자료형으로 받습니다.

구조화된 결과는 다음 작업에 그대로 사용됩니다.

1. `lines`의 화자·대사·감정·행동·숨은 의도를 화면에 각각 나누어 표시합니다.
2. `continuity`를 이용해 설정 충돌 여부와 경고를 확인합니다.
3. `storyUpdates.eventSummary`를 사건 연대기에 저장합니다.
4. `relationshipChanges`를 캐릭터 관계 기록에 반영합니다.
5. `fateSignals`를 필연의 누적 진행도에 반영합니다.
6. 저장된 결과를 다음 장면의 기억 Tool이 다시 조회합니다.

또한 Pydantic이 필수 필드, 문자열·숫자 자료형, 대사 개수와 진행도 범위를 검증하므로 잘못된 형식의 응답이 애플리케이션 상태에 바로 저장되는 것을 막을 수 있습니다. Structured Output은 출력 형식을 꾸미기 위한 기능이 아니라 LLM의 응답과 프로그램의 저장·갱신 로직을 연결하는 데이터 계약입니다.

### 3. Middleware를 사용한 이유
사용자에게 가장 중요한 경험은 세계관과 장면 요청을 작성한 뒤 **장면 생성 버튼을 한 번 눌러 결과를 받는 것**입니다. 사용자는 장면이 내부적으로 작가 Agent와 검수 Agent를 거친다는 사실이나, 어느 모델 호출에서 일시적인 연결 오류가 발생했는지를 알 필요가 없습니다.

하지만 실제 AI 서비스에서는 순간적인 네트워크 불안정, API 서버 오류 또는 요청 집중으로 모델 호출이 한 번 실패할 수 있습니다. Middleware가 없다면 사용자는 실패 메시지를 보고 같은 버튼을 다시 누르거나 장면 요청을 다시 시도해야 합니다. 이 과정은 창작 흐름을 끊고 서비스가 불안정하다는 인상을 줍니다.

현재 노트북은 `ModelRetryMiddleware`를 작가 Agent와 검수 Agent에 적용합니다. 모델 호출이 일시적으로 실패하면 사용자에게 바로 실패를 보여주지 않고 내부에서 잠시 기다린 뒤 한 번 자동으로 재시도합니다. 따라서 사용자는 별도의 조작 없이 처음 누른 장면 생성 요청으로 결과를 받을 가능성이 높아집니다.

```text
사용자가 장면 생성 요청
        ↓
일시적인 모델 호출 오류 발생
        ↓
ModelRetryMiddleware가 내부에서 자동 재시도
        ↓
사용자는 버튼을 다시 누르지 않고 장면 결과 확인
```

한 번의 재시도가 모든 장애를 완전히 없애는 것은 아니지만, 짧은 일시 오류 때문에 사용자가 작업을 반복해야 하는 상황을 줄여줍니다. 즉, Middleware를 사용한 목적은 기술을 추가하기 위한 것이 아니라 **사용자의 한 번의 요청이 가능한 한 끊기지 않고 최종 장면 출력까지 이어지게 하기 위해서**입니다. 이야기의 사건·관계·필연 갱신은 Middleware가 아니라 최종 결과가 승인된 뒤 `_apply_story_updates()`에서 별도로 처리합니다.